<a href="https://colab.research.google.com/github/yadavanshudstcims2025-cell/causal-ml-discount-effect/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install econml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 64.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [econml]


In [ ]:
# ============================================================
# STEP 1: GENERATE SYNTHETIC CAUSAL DATA
# ============================================================
#
# Objective:
# Create a realistic observational customer dataset in which:
#
# 1. Discount assignment is CONFOUNDED.
# 2. Customers have different characteristics.
# 3. The TRUE treatment effect varies across customers.
# 4. The average TRUE treatment effect is approximately 0.015.
#
# This allows us to study:
#       - Naive estimation
#       - Causal Forest
#       - Uplift modeling
#       - Statistical validation
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

np.random.seed(42)


# ------------------------------------------------------------
# 2. Number of customers
# ------------------------------------------------------------

n_customers = 10_000

# Create data directory
os.makedirs("data", exist_ok=True)


# ------------------------------------------------------------
# 3. Generate customer characteristics
# ------------------------------------------------------------
#
# These variables represent information a company might
# observe about its customers.
# ------------------------------------------------------------

data = pd.DataFrame({

    # Number of purchases made previously
    "previous_purchases":
        np.random.randint(0, 50, n_customers),

    # Typical amount spent per order
    "avg_order_value":
        np.random.uniform(10, 500, n_customers),

    # Number of days since the customer last purchased
    "days_since_last_purchase":
        np.random.randint(1, 365, n_customers),

    # Whether the customer clicked a marketing email
    # Approximately 35% are assumed to click
    "email_clicked":
        np.random.binomial(1, 0.35, n_customers)
})


# ============================================================
# 4. CONFONDED TREATMENT ASSIGNMENT
# ============================================================
#
# Treatment:
#     1 = customer receives discount
#     0 = customer does not receive discount
#
# Assumption:
# High-value customers are more likely to receive discounts.
#
# Therefore:
#
# Average order value
#        ↓
#   Discount assignment
#
# and also:
#
# Average order value
#        ↓
#   Purchase probability
#
# This creates CONFOUNDING.
# ============================================================

prob_treatment = (
    0.20
    + 0.50 * (
        data["avg_order_value"] / 500
    )
)

# Ensure probabilities remain between 0 and 1
prob_treatment = np.clip(
    prob_treatment,
    0.05,
    0.90
)

# Randomly assign treatment according to probability
data["treatment"] = (
    np.random.random(n_customers)
    < prob_treatment
).astype(int)


# ============================================================
# 5. BASELINE PURCHASE PROBABILITY
# ============================================================
#
# This represents the probability of purchase WITHOUT
# receiving a discount.
#
# Customer characteristics affect this probability.
# ============================================================

baseline_prob = (
    0.10
    + 0.002 * data["previous_purchases"]
    + 0.00015 * data["avg_order_value"]
    + 0.05 * data["email_clicked"]
)


# Keep probabilities within a valid range
baseline_prob = np.clip(
    baseline_prob,
    0.01,
    0.80
)


# ============================================================
# 6. CREATE HETEROGENEOUS TRUE TREATMENT EFFECT
# ============================================================
#
# IMPORTANT DIFFERENCE FROM THE PREVIOUS VERSION:
#
# Previously:
#
#     treatment_effect = 0.015
#
# for EVERY customer.
#
# Now each customer gets a slightly different treatment
# effect.
#
# The effect depends on customer characteristics.
#
# This allows Causal Forest and uplift modeling to study
# heterogeneous treatment effects.
# ============================================================

raw_effect = (
    0.005

    + 0.00015 * data["previous_purchases"]

    + 0.000015 * data["avg_order_value"]

    + 0.002 * data["email_clicked"]
)


# ------------------------------------------------------------
# 7. Scale effects so that the AVERAGE true effect = 0.015
# ------------------------------------------------------------
#
# We want the overall average treatment effect to remain
# approximately 0.015.
#
# This means:
#
#     Average TRUE individual effect ≈ 0.015
#
# while individual effects differ.
# ------------------------------------------------------------

target_ate = 0.015

scaling_factor = (
    target_ate / raw_effect.mean()
)

true_individual_effect = (
    raw_effect * scaling_factor
)


# ------------------------------------------------------------
# 8. Store the TRUE individual treatment effect
# ------------------------------------------------------------
#
# This column is normally unavailable in real-world data.
# We keep it because this is a simulation and it allows us
# to evaluate whether our ML methods recover the truth.
# ------------------------------------------------------------

data["true_individual_effect"] = (
    true_individual_effect
)


# ============================================================
# 9. CREATE POTENTIAL OUTCOME PROBABILITIES
# ============================================================
#
# Y(0):
# Purchase probability WITHOUT discount
#
# Y(1):
# Purchase probability WITH discount
#
# Causal effect:
#
#     Y(1) - Y(0)
#
# = true_individual_effect
# ============================================================

prob_without_discount = baseline_prob

prob_with_discount = (
    baseline_prob
    + true_individual_effect
)


# Keep probabilities valid
prob_with_discount = np.clip(
    prob_with_discount,
    0.01,
    0.95
)


# ============================================================
# 10. CREATE OBSERVED PURCHASE OUTCOME
# ============================================================
#
# We only observe ONE potential outcome for each customer.
#
# If treatment = 1:
#     observe Y(1)
#
# If treatment = 0:
#     observe Y(0)
# ============================================================

observed_probability = np.where(
    data["treatment"] == 1,
    prob_with_discount,
    prob_without_discount
)


# Generate binary purchase outcome
data["outcome"] = (
    np.random.random(n_customers)
    < observed_probability
).astype(int)


# ============================================================
# 11. SAVE DATA
# ============================================================

file_path = "data/synthetic_data.csv"

data.to_csv(
    file_path,
    index=False
)


# ============================================================
# 12. CALCULATE TRUE ATE
# ============================================================
#
# Because we created the true individual effects ourselves,
# we can calculate the actual average causal effect.
# ============================================================

true_ate = (
    data["true_individual_effect"].mean()
)


# ============================================================
# 13. PRINT RESULTS
# ============================================================

print("=" * 65)
print("STEP 1: SYNTHETIC CAUSAL DATA GENERATED")
print("=" * 65)

print(
    f"Number of customers: "
    f"{n_customers}"
)

print(
    f"Treatment rate: "
    f"{data['treatment'].mean():.2%}"
)

print(
    f"Overall purchase rate: "
    f"{data['outcome'].mean():.2%}"
)

print("-" * 65)

print(
    f"Purchase rate - No discount: "
    f"{data.loc[data['treatment'] == 0, 'outcome'].mean():.2%}"
)

print(
    f"Purchase rate - Discount: "
    f"{data.loc[data['treatment'] == 1, 'outcome'].mean():.2%}"
)

print("-" * 65)

print(
    f"TRUE ATE: "
    f"{true_ate:.4f}"
)

print(
    f"TRUE ATE in percentage points: "
    f"{true_ate * 100:.2f}%"
)

print("-" * 65)

print(
    f"Minimum true individual effect: "
    f"{data['true_individual_effect'].min():.4f}"
)

print(
    f"Maximum true individual effect: "
    f"{data['true_individual_effect'].max():.4f}"
)

print(
    f"Median true individual effect: "
    f"{data['true_individual_effect'].median():.4f}"
)

print("-" * 65)

# Confounding check
treated_aov = data.loc[
    data["treatment"] == 1,
    "avg_order_value"
].mean()

control_aov = data.loc[
    data["treatment"] == 0,
    "avg_order_value"
].mean()

print("CONFOUNDING CHECK")

print(
    f"Average order value - Treated: "
    f"{treated_aov:.2f}"
)

print(
    f"Average order value - Control: "
    f"{control_aov:.2f}"
)

print("=" * 65)

print(
    f"Data saved to: {file_path}"
)


STEP 1: SYNTHETIC CAUSAL DATA GENERATED
Number of customers: 10000
Treatment rate: 45.42%
Overall purchase rate: 21.01%
-----------------------------------------------------------------
Purchase rate - No discount: 20.06%
Purchase rate - Discount: 22.15%
-----------------------------------------------------------------
TRUE ATE: 0.0150
TRUE ATE in percentage points: 1.50%
-----------------------------------------------------------------
Minimum true individual effect: 0.0059
Maximum true individual effect: 0.0248
Median true individual effect: 0.0150
-----------------------------------------------------------------
CONFOUNDING CHECK
Average order value - Treated: 296.24
Average order value - Control: 217.49
Data saved to: data/synthetic_data.csv


In [ ]:
# ============================================================
# STEP 2: NAIVE ESTIMATION
# ============================================================

import pandas as pd
import numpy as np

# Load the NEW data generated in Step 1
data = pd.read_csv("data/synthetic_data.csv")


# ------------------------------------------------------------
# 1. Separate treatment groups
# ------------------------------------------------------------

treated = data[data["treatment"] == 1]

control = data[data["treatment"] == 0]


# ------------------------------------------------------------
# 2. Calculate observed purchase rates
# ------------------------------------------------------------

treated_mean = treated["outcome"].mean()

control_mean = control["outcome"].mean()


# ------------------------------------------------------------
# 3. Naive ATE
# ------------------------------------------------------------

naive_ate = (
    treated_mean
    - control_mean
)


# ------------------------------------------------------------
# 4. TRUE ATE
# ------------------------------------------------------------
#
# Unlike the previous version, we don't hard-code 0.015.
# We calculate it from the true individual effects created
# in Step 1.
# ------------------------------------------------------------

true_ate = (
    data["true_individual_effect"].mean()
)


# ------------------------------------------------------------
# 5. Calculate bias
# ------------------------------------------------------------

bias = naive_ate - true_ate

absolute_error = abs(bias)

relative_error = (
    absolute_error / abs(true_ate)
) * 100


# ------------------------------------------------------------
# 6. Confounding check
# ------------------------------------------------------------

treated_aov = treated["avg_order_value"].mean()

control_aov = control["avg_order_value"].mean()


# ------------------------------------------------------------
# 7. Print results
# ------------------------------------------------------------

print("=" * 65)
print("STEP 2: NAIVE ESTIMATION")
print("=" * 65)

print(
    f"Treated customers: "
    f"{len(treated)}"
)

print(
    f"Control customers: "
    f"{len(control)}"
)

print("-" * 65)

print(
    f"Treated purchase rate: "
    f"{treated_mean:.4f}"
)

print(
    f"Control purchase rate: "
    f"{control_mean:.4f}"
)

print("-" * 65)

print(
    f"Naive ATE: "
    f"{naive_ate:.4f}"
)

print(
    f"True ATE: "
    f"{true_ate:.4f}"
)

print(
    f"Bias: "
    f"{bias:.4f}"
)

print(
    f"Absolute error: "
    f"{absolute_error:.4f}"
)

print(
    f"Relative error: "
    f"{relative_error:.2f}%"
)

print("-" * 65)

print("CONFOUNDING CHECK")

print(
    f"Average order value - Treated: "
    f"{treated_aov:.2f}"
)

print(
    f"Average order value - Control: "
    f"{control_aov:.2f}"
)

print("=" * 65)


# ------------------------------------------------------------
# 8. Interpretation
# ------------------------------------------------------------

print("\nINTERPRETATION")
print("=" * 65)

print(
    "The naive estimator compares the observed purchase "
    "rates of discounted and non-discounted customers."
)

print(
    "Because high-value customers are more likely to receive "
    "discounts, treatment assignment is confounded."
)

print(
    f"The naive estimate is {naive_ate:.4f}, "
    f"while the true ATE is {true_ate:.4f}."
)

if naive_ate > true_ate:
    print(
        "Therefore, the naive method overestimates the "
        "average causal effect of the discount."
    )
else:
    print(
        "Therefore, the naive method underestimates the "
        "average causal effect of the discount."
    )

STEP 2: NAIVE ESTIMATION
Treated customers: 4542
Control customers: 5458
-----------------------------------------------------------------
Treated purchase rate: 0.2215
Control purchase rate: 0.2006
-----------------------------------------------------------------
Naive ATE: 0.0209
True ATE: 0.0150
Bias: 0.0059
Absolute error: 0.0059
Relative error: 39.10%
-----------------------------------------------------------------
CONFOUNDING CHECK
Average order value - Treated: 296.24
Average order value - Control: 217.49

INTERPRETATION
The naive estimator compares the observed purchase rates of discounted and non-discounted customers.
Because high-value customers are more likely to receive discounts, treatment assignment is confounded.
The naive estimate is 0.0209, while the true ATE is 0.0150.
Therefore, the naive method overestimates the average causal effect of the discount.


In [ ]:
# ============================================================
# STEP 3: CAUSAL FOREST
# ============================================================
#
# Objective:
# Estimate:
#
# 1. Average Treatment Effect (ATE)
# 2. Individual Treatment Effects (CATE)
# 3. Accuracy of estimated CATE compared with TRUE CATE
#
# Step 2 showed:
# Naive ATE = 0.0209
# True ATE  = 0.0150
#
# Now we use Causal Forest to account for customer
# characteristics and estimate heterogeneous treatment effects.
# ============================================================

import pandas as pd
import numpy as np

from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

data = pd.read_csv("data/synthetic_data.csv")


# ------------------------------------------------------------
# 2. Define X, T and Y
# ------------------------------------------------------------

feature_columns = [
    "previous_purchases",
    "avg_order_value",
    "days_since_last_purchase",
    "email_clicked"
]

# Customer characteristics
X = data[feature_columns]

# Treatment
# 1 = discount
# 0 = no discount
T = data["treatment"].astype(float)

# Observed outcome
# 1 = purchase
# 0 = no purchase
Y = data["outcome"].astype(float)


# ------------------------------------------------------------
# 3. TRUE treatment effects
# ------------------------------------------------------------
#
# This is available because we created synthetic data.
#
# In real-world data, these values would NOT be observed.
# ------------------------------------------------------------

true_cate = data["true_individual_effect"].values

true_ate = true_cate.mean()


# ------------------------------------------------------------
# 4. Create Causal Forest DML
# ------------------------------------------------------------

print("Training Causal Forest...")

cf = CausalForestDML(

    # Outcome model
    model_y=RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=10,
        random_state=42
    ),

    # Treatment model
    model_t=RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=10,
        random_state=42
    ),

    # Number of causal trees
    n_estimators=300,

    # Minimum observations per leaf
    min_samples_leaf=10,

    # Maximum tree depth
    max_depth=5,

    random_state=42
)


# ------------------------------------------------------------
# 5. Fit the Causal Forest
# ------------------------------------------------------------

cf.fit(
    Y,
    T,
    X=X
)


# ------------------------------------------------------------
# 6. Estimate individual treatment effects
# ------------------------------------------------------------

estimated_cate = cf.effect(X)


# ------------------------------------------------------------
# 7. Estimate ATE
# ------------------------------------------------------------

estimated_ate = estimated_cate.mean()


# ------------------------------------------------------------
# 8. ATE error
# ------------------------------------------------------------

ate_error = (
    estimated_ate - true_ate
)

ate_absolute_error = abs(ate_error)

ate_relative_error = (
    ate_absolute_error /
    abs(true_ate)
) * 100


# ------------------------------------------------------------
# 9. Naive estimator for comparison
# ------------------------------------------------------------

naive_ate = (
    data.loc[
        data["treatment"] == 1,
        "outcome"
    ].mean()
    -
    data.loc[
        data["treatment"] == 0,
        "outcome"
    ].mean()
)

naive_absolute_error = abs(
    naive_ate - true_ate
)

naive_relative_error = (
    naive_absolute_error /
    abs(true_ate)
) * 100


# ------------------------------------------------------------
# 10. CATE RMSE
# ------------------------------------------------------------
#
# Compare estimated individual treatment effects against
# the TRUE individual treatment effects.
# ------------------------------------------------------------

cate_rmse = np.sqrt(
    mean_squared_error(
        true_cate,
        estimated_cate
    )
)


# ------------------------------------------------------------
# 11. Correlation between TRUE and estimated CATE
# ------------------------------------------------------------

cate_correlation = np.corrcoef(
    true_cate,
    estimated_cate
)[0, 1]


# ------------------------------------------------------------
# 12. Add estimated CATE to dataset
# ------------------------------------------------------------

data["estimated_cate"] = estimated_cate


# ------------------------------------------------------------
# 13. Calculate CATE error for every customer
# ------------------------------------------------------------

data["cate_error"] = (
    data["estimated_cate"]
    -
    data["true_individual_effect"]
)


# ============================================================
# 14. PRINT ATE RESULTS
# ============================================================

print("=" * 65)
print("STEP 3: CAUSAL FOREST RESULTS")
print("=" * 65)

print(
    f"True ATE:              "
    f"{true_ate:.4f}"
)

print(
    f"Naive ATE:              "
    f"{naive_ate:.4f}"
)

print(
    f"Causal Forest ATE:     "
    f"{estimated_ate:.4f}"
)

print("-" * 65)

print(
    f"Naive absolute error:   "
    f"{naive_absolute_error:.4f}"
)

print(
    f"Causal Forest absolute error: "
    f"{ate_absolute_error:.4f}"
)

print("-" * 65)

print(
    f"Naive relative error:   "
    f"{naive_relative_error:.2f}%"
)

print(
    f"Causal Forest relative error: "
    f"{ate_relative_error:.2f}%"
)

print("=" * 65)


# ============================================================
# 15. HETEROGENEOUS EFFECT RESULTS
# ============================================================

print("\nHETEROGENEOUS TREATMENT EFFECTS")
print("=" * 65)

print(
    f"True minimum effect: "
    f"{true_cate.min():.4f}"
)

print(
    f"True maximum effect: "
    f"{true_cate.max():.4f}"
)

print(
    f"Estimated minimum effect: "
    f"{estimated_cate.min():.4f}"
)

print(
    f"Estimated maximum effect: "
    f"{estimated_cate.max():.4f}"
)

print("-" * 65)

print(
    f"CATE RMSE: "
    f"{cate_rmse:.4f}"
)

print(
    f"True vs estimated CATE correlation: "
    f"{cate_correlation:.4f}"
)


# ============================================================
# 16. TOP CUSTOMERS
# ============================================================

print("\nTOP 10 CUSTOMERS BY ESTIMATED TREATMENT EFFECT")
print("=" * 65)

top_customers = (
    data
    .sort_values(
        "estimated_cate",
        ascending=False
    )
    .head(10)
)

print(
    top_customers[
        feature_columns
        +
        [
            "true_individual_effect",
            "estimated_cate"
        ]
    ].to_string(index=False)
)


# ============================================================
# 17. BUSINESS INTERPRETATION
# ============================================================

print("\nBUSINESS INTERPRETATION")
print("=" * 65)

if ate_absolute_error < naive_absolute_error:

    print(
        "Causal Forest produced an ATE closer to the "
        "true ATE than the naive estimator."
    )

else:

    print(
        "Causal Forest did not produce an ATE closer "
        "to the true ATE than the naive estimator."
    )


if cate_correlation > 0:

    print(
        "The positive CATE correlation indicates that "
        "the model is capturing some of the differences "
        "in treatment response across customers."
    )

else:

    print(
        "The CATE correlation is not positive, indicating "
        "that the model is not reliably identifying "
        "individual treatment-effect differences."
    )

print(
    "These estimated individual effects can be used "
    "in Step 4 to target customers who are predicted "
    "to benefit most from discounts."
)

Training Causal Forest...
STEP 3: CAUSAL FOREST RESULTS
True ATE:              0.0150
Naive ATE:              0.0209
Causal Forest ATE:     0.0102
-----------------------------------------------------------------
Naive absolute error:   0.0059
Causal Forest absolute error: 0.0048
-----------------------------------------------------------------
Naive relative error:   39.10%
Causal Forest relative error: 32.09%

HETEROGENEOUS TREATMENT EFFECTS
True minimum effect: 0.0059
True maximum effect: 0.0248
Estimated minimum effect: -0.0751
Estimated maximum effect: 0.1000
-----------------------------------------------------------------
CATE RMSE: 0.0201
True vs estimated CATE correlation: 0.1755

TOP 10 CUSTOMERS BY ESTIMATED TREATMENT EFFECT
 previous_purchases  avg_order_value  days_since_last_purchase  email_clicked  true_individual_effect  estimated_cate
                 41       496.746819                        29              0                0.021123        0.100032
                 4

In [ ]:
# ============================================================
# STEP 4: CROSS-FITTED TWO-MODEL UPLIFT MODELING
# ============================================================
#
# Objective:
# Identify customers who are most likely to benefit from
# receiving a discount.
#
# We use two Random Forest models:
#
# Model 1:
# P(Y=1 | X, Discount)
#
# Model 2:
# P(Y=1 | X, No Discount)
#
# Predicted Uplift =
#
# P(Y=1 | X, Discount)
# -
# P(Y=1 | X, No Discount)
#
# IMPORTANT:
# We use out-of-fold predictions so that each customer's
# prediction is produced by a model that did NOT train
# directly on that customer's outcome.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error


# ============================================================
# 1. Create output folders
# ============================================================

os.makedirs("results/plots", exist_ok=True)


# ============================================================
# 2. Load data
# ============================================================

data = pd.read_csv(
    "data/synthetic_data.csv"
)


# ============================================================
# 3. Define variables
# ============================================================

feature_columns = [
    "previous_purchases",
    "avg_order_value",
    "days_since_last_purchase",
    "email_clicked"
]

X = data[feature_columns]

T = data["treatment"].values

Y = data["outcome"].values

true_effect = data[
    "true_individual_effect"
].values


# ============================================================
# 4. Create arrays for out-of-fold predictions
# ============================================================

predicted_purchase_if_treated = np.zeros(
    len(data)
)

predicted_purchase_if_control = np.zeros(
    len(data)
)


# ============================================================
# 5. Five-fold cross-fitting
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


print("=" * 65)
print("STEP 4: CROSS-FITTED TWO-MODEL UPLIFT MODEL")
print("=" * 65)

print("\nTraining 5-fold uplift models...")


for fold, (train_index, test_index) in enumerate(
    skf.split(X, T),
    start=1
):

    print(f"Processing fold {fold}/5...")


    # --------------------------------------------------------
    # Training data
    # --------------------------------------------------------

    X_train = X.iloc[train_index]

    T_train = T[train_index]

    Y_train = Y[train_index]


    # --------------------------------------------------------
    # Validation data
    # --------------------------------------------------------

    X_test = X.iloc[test_index]


    # --------------------------------------------------------
    # Treatment training data
    # --------------------------------------------------------

    treatment_mask = (
        T_train == 1
    )

    X_treatment = X_train[
        treatment_mask
    ]

    Y_treatment = Y_train[
        treatment_mask
    ]


    # --------------------------------------------------------
    # Control training data
    # --------------------------------------------------------

    control_mask = (
        T_train == 0
    )

    X_control = X_train[
        control_mask
    ]

    Y_control = Y_train[
        control_mask
    ]


    # ========================================================
    # MODEL 1: TREATMENT MODEL
    # ========================================================

    treatment_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=30,
        random_state=42 + fold,
        n_jobs=-1
    )

    treatment_model.fit(
        X_treatment,
        Y_treatment
    )


    # ========================================================
    # MODEL 2: CONTROL MODEL
    # ========================================================

    control_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=30,
        random_state=100 + fold,
        n_jobs=-1
    )

    control_model.fit(
        X_control,
        Y_control
    )


    # ========================================================
    # Predict on validation fold
    # ========================================================

    predicted_purchase_if_treated[
        test_index
    ] = treatment_model.predict_proba(
        X_test
    )[:, 1]


    predicted_purchase_if_control[
        test_index
    ] = control_model.predict_proba(
        X_test
    )[:, 1]


# ============================================================
# 6. Calculate cross-fitted uplift
# ============================================================

predicted_uplift = (
    predicted_purchase_if_treated
    -
    predicted_purchase_if_control
)


# Add predictions to dataset

data[
    "purchase_if_discount"
] = predicted_purchase_if_treated


data[
    "purchase_if_no_discount"
] = predicted_purchase_if_control


data[
    "predicted_uplift"
] = predicted_uplift


# ============================================================
# 7. Overall uplift performance
# ============================================================

uplift_rmse = np.sqrt(
    mean_squared_error(
        true_effect,
        predicted_uplift
    )
)

uplift_correlation = np.corrcoef(
    true_effect,
    predicted_uplift
)[0, 1]


print("\n" + "=" * 65)
print("UPLIFT MODEL RESULTS")
print("=" * 65)

print(
    f"Mean predicted uplift: "
    f"{predicted_uplift.mean():.4f}"
)

print(
    f"Mean true treatment effect: "
    f"{true_effect.mean():.4f}"
)

print("-" * 65)

print(
    f"Uplift RMSE: "
    f"{uplift_rmse:.4f}"
)

print(
    f"True vs predicted uplift correlation: "
    f"{uplift_correlation:.4f}"
)


# ============================================================
# 8. Rank customers
# ============================================================

data = data.sort_values(
    "predicted_uplift",
    ascending=False
).reset_index(
    drop=True
)


data[
    "target_rank"
] = np.arange(
    1,
    len(data) + 1
)


# ============================================================
# 9. Create targeting percentiles
# ============================================================
#
# We examine:
#
# Top 10%
# Top 20%
# Top 30%
# Top 40%
# ...
# Top 100%
#
# For each group we calculate the TRUE average treatment
# effect.
# ============================================================

percentages = [
    10, 20, 30, 40, 50,
    60, 70, 80, 90, 100
]


targeting_results = []


for percentage in percentages:

    n_target = int(
        len(data) * percentage / 100
    )

    selected = data.iloc[
        :n_target
    ]


    mean_true_effect = (
        selected[
            "true_individual_effect"
        ].mean()
    )


    mean_predicted_uplift = (
        selected[
            "predicted_uplift"
        ].mean()
    )


    targeting_results.append({

        "percentage_targeted":
            percentage,

        "customers_targeted":
            n_target,

        "mean_predicted_uplift":
            mean_predicted_uplift,

        "mean_true_effect":
            mean_true_effect
    })


targeting_results = pd.DataFrame(
    targeting_results
)


# ============================================================
# 10. Print targeting performance
# ============================================================

print("\n" + "=" * 65)
print("TARGETING PERFORMANCE")
print("=" * 65)

print(
    targeting_results.to_string(
        index=False
    )
)


# ============================================================
# 11. Top 20 customers
# ============================================================

print("\n" + "=" * 65)
print("TOP 20 CUSTOMERS FOR DISCOUNT TARGETING")
print("=" * 65)

print(
    data[
        [
            "target_rank",
            "previous_purchases",
            "avg_order_value",
            "days_since_last_purchase",
            "email_clicked",
            "true_individual_effect",
            "predicted_uplift"
        ]
    ].head(20).to_string(
        index=False
    )
)


# ============================================================
# 12. Business targeting groups
# ============================================================

top_10 = data.iloc[
    :int(0.10 * len(data))
]

top_30 = data.iloc[
    :int(0.30 * len(data))
]

remaining = data.iloc[
    int(0.30 * len(data)) :
]


print("\n" + "=" * 65)
print("BUSINESS TARGETING COMPARISON")
print("=" * 65)

print(
    f"Top 10% true average effect: "
    f"{top_10['true_individual_effect'].mean():.4f}"
)

print(
    f"Top 30% true average effect: "
    f"{top_30['true_individual_effect'].mean():.4f}"
)

print(
    f"Remaining 70% true average effect: "
    f"{remaining['true_individual_effect'].mean():.4f}"
)


# ============================================================
# 13. Calculate targeting lift
# ============================================================

overall_true_ate = (
    data[
        "true_individual_effect"
    ].mean()
)

top_10_true_effect = (
    top_10[
        "true_individual_effect"
    ].mean()
)

targeting_lift = (
    top_10_true_effect
    -
    overall_true_ate
)


print(
    f"\nTop 10% targeting lift over "
    f"overall ATE: {targeting_lift:.4f}"
)


# ============================================================
# 14. Visualization 1
#     True vs predicted uplift
# ============================================================

plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    true_effect,
    predicted_uplift,
    alpha=0.30
)

minimum = min(
    true_effect.min(),
    predicted_uplift.min()
)

maximum = max(
    true_effect.max(),
    predicted_uplift.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel(
    "True Individual Treatment Effect"
)

plt.ylabel(
    "Predicted Uplift"
)

plt.title(
    "True vs Predicted Individual Treatment Effect"
)

plt.tight_layout()

plt.savefig(
    "results/plots/04_true_vs_predicted_uplift.png",
    dpi=300
)

plt.close()


# ============================================================
# 15. Visualization 2
#     Targeting curve
# ============================================================

plt.figure(
    figsize=(8, 6)
)

plt.plot(
    targeting_results[
        "percentage_targeted"
    ],
    targeting_results[
        "mean_true_effect"
    ],
    marker="o"
)

plt.axhline(
    y=overall_true_ate,
    linestyle="--",
    label="Overall True ATE"
)

plt.xlabel(
    "Percentage of Customers Targeted"
)

plt.ylabel(
    "Average True Treatment Effect"
)

plt.title(
    "Uplift Targeting Curve"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "results/plots/05_uplift_targeting_curve.png",
    dpi=300
)

plt.close()


# ============================================================
# 16. Visualization 3
#     Predicted uplift distribution
# ============================================================

plt.figure(
    figsize=(8, 6)
)

plt.hist(
    predicted_uplift,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    x=0,
    linestyle="--"
)

plt.xlabel(
    "Predicted Uplift"
)

plt.ylabel(
    "Number of Customers"
)

plt.title(
    "Distribution of Predicted Customer Uplift"
)

plt.tight_layout()

plt.savefig(
    "results/plots/06_uplift_distribution.png",
    dpi=300
)

plt.close()


# ============================================================
# 17. Save targeting results
# ============================================================

data[
    feature_columns
    + [
        "treatment",
        "outcome",
        "true_individual_effect",
        "purchase_if_discount",
        "purchase_if_no_discount",
        "predicted_uplift",
        "target_rank"
    ]
].to_csv(
    "results/uplift_targeting_results.csv",
    index=False
)


targeting_results.to_csv(
    "results/targeting_performance.csv",
    index=False
)


# ============================================================
# 18. Final interpretation
# ============================================================

print("\n" + "=" * 65)
print("STEP 4 COMPLETED")
print("=" * 65)

if (
    top_10_true_effect
    > overall_true_ate
):

    print(
        "The top 10% targeted customers have a "
        "higher average true treatment effect than "
        "the overall population."
    )

else:

    print(
        "The top 10% targeted customers do not have "
        "a higher average true treatment effect than "
        "the overall population."
    )


print(
    "\nTargeting results saved to:"
)

print(
    "results/uplift_targeting_results.csv"
)

print(
    "results/targeting_performance.csv"
)

print("\nPlots saved to:")

print(
    "results/plots/04_true_vs_predicted_uplift.png"
)

print(
    "results/plots/05_uplift_targeting_curve.png"
)

print(
    "results/plots/06_uplift_distribution.png"
)

print("=" * 65)

STEP 4: CROSS-FITTED TWO-MODEL UPLIFT MODEL

Training 5-fold uplift models...
Processing fold 1/5...
Processing fold 2/5...
Processing fold 3/5...
Processing fold 4/5...
Processing fold 5/5...

UPLIFT MODEL RESULTS
Mean predicted uplift: 0.0093
Mean true treatment effect: 0.0150
-----------------------------------------------------------------
Uplift RMSE: 0.0413
True vs predicted uplift correlation: -0.0591

TARGETING PERFORMANCE
 percentage_targeted  customers_targeted  mean_predicted_uplift  mean_true_effect
                  10                1000               0.077929          0.015460
                  20                2000               0.064224          0.014984
                  30                3000               0.055124          0.014826
                  40                4000               0.047935          0.014759
                  50                5000               0.041614          0.014700
                  60                6000               0.035612          

In [ ]:
# ============================================================
# STEP 5: STATISTICAL VALIDATION
# ============================================================
#
# This step contains:
#
# 1. Bootstrap confidence interval for ATE
# 2. Hypothesis testing
# 3. Placebo test
# 4. Final visualizations
#
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.linear_model import LinearRegression


# ============================================================
# 1. Create output directory
# ============================================================

os.makedirs(
    "results/plots",
    exist_ok=True
)


# ============================================================
# 2. Load data
# ============================================================

data = pd.read_csv(
    "data/synthetic_data.csv"
)


# ============================================================
# 3. Define variables
# ============================================================

feature_columns = [
    "previous_purchases",
    "avg_order_value",
    "days_since_last_purchase",
    "email_clicked"
]

X = data[feature_columns]

T = data["treatment"].values

Y = data["outcome"].values


# ============================================================
# 4. NAIVE ATE
# ============================================================

treated_mean = Y[T == 1].mean()

control_mean = Y[T == 0].mean()

naive_ate = (
    treated_mean -
    control_mean
)


# ============================================================
# 5. TRUE ATE
# ============================================================

true_ate = data[
    "true_individual_effect"
].mean()


print("=" * 65)
print("STEP 5: STATISTICAL VALIDATION")
print("=" * 65)

print(
    f"Naive ATE: {naive_ate:.4f}"
)

print(
    f"True ATE:  {true_ate:.4f}"
)


# ============================================================
# PART A: BOOTSTRAP CONFIDENCE INTERVAL
# ============================================================
#
# We repeatedly resample customers with replacement.
#
# For every bootstrap sample:
#
# ATE = mean(Y | T=1)
#       -
#       mean(Y | T=0)
#
# Then we use the 2.5th and 97.5th percentiles
# to construct a 95% bootstrap confidence interval.
# ============================================================

print("\n" + "=" * 65)
print("PART A: BOOTSTRAP CONFIDENCE INTERVAL")
print("=" * 65)


n_bootstrap = 500

bootstrap_ates = []

rng = np.random.default_rng(42)


for i in range(n_bootstrap):

    # Randomly sample customers with replacement
    sample_indices = rng.choice(
        len(data),
        size=len(data),
        replace=True
    )

    sample = data.iloc[
        sample_indices
    ]


    # Make sure both treatment groups exist
    if (
        sample["treatment"].nunique()
        < 2
    ):
        continue


    treated = sample[
        sample["treatment"] == 1
    ]["outcome"].mean()


    control = sample[
        sample["treatment"] == 0
    ]["outcome"].mean()


    bootstrap_ate = (
        treated -
        control
    )


    bootstrap_ates.append(
        bootstrap_ate
    )


bootstrap_ates = np.array(
    bootstrap_ates
)


# ============================================================
# Calculate 95% confidence interval
# ============================================================

lower_ci = np.percentile(
    bootstrap_ates,
    2.5
)

upper_ci = np.percentile(
    bootstrap_ates,
    97.5
)


bootstrap_mean = (
    bootstrap_ates.mean()
)


print(
    f"Bootstrap replications: "
    f"{len(bootstrap_ates)}"
)

print(
    f"Bootstrap mean ATE: "
    f"{bootstrap_mean:.4f}"
)

print(
    f"95% Bootstrap CI: "
    f"[{lower_ci:.4f}, {upper_ci:.4f}]"
)


# ============================================================
# Interpretation
# ============================================================

if (
    lower_ci <= 0
    and upper_ci >= 0
):

    print(
        "The confidence interval contains zero."
    )

else:

    print(
        "The confidence interval does not contain zero."
    )


# ============================================================
# Bootstrap visualization
# ============================================================

plt.figure(
    figsize=(9, 6)
)

plt.hist(
    bootstrap_ates,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    lower_ci,
    linestyle="--",
    linewidth=2,
    label=f"2.5% = {lower_ci:.4f}"
)

plt.axvline(
    upper_ci,
    linestyle="--",
    linewidth=2,
    label=f"97.5% = {upper_ci:.4f}"
)

plt.axvline(
    true_ate,
    linestyle="-",
    linewidth=2,
    label=f"True ATE = {true_ate:.4f}"
)

plt.xlabel(
    "Bootstrap ATE"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Bootstrap Distribution of ATE"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "results/plots/07_bootstrap_ate.png",
    dpi=300
)

plt.close()


# ============================================================
# PART B: HYPOTHESIS TEST
# ============================================================
#
# H0: ATE = 0
#
# H1: ATE != 0
#
# We use a two-sample test for the observed purchase rates.
# ============================================================

print("\n" + "=" * 65)
print("PART B: HYPOTHESIS TEST")
print("=" * 65)


treated_outcomes = data[
    data["treatment"] == 1
]["outcome"]

control_outcomes = data[
    data["treatment"] == 0
]["outcome"]


# Welch's t-test
t_statistic, p_value = stats.ttest_ind(
    treated_outcomes,
    control_outcomes,
    equal_var=False
)


print(
    f"T-statistic: "
    f"{t_statistic:.4f}"
)

print(
    f"P-value: "
    f"{p_value:.6f}"
)


alpha = 0.05


if p_value < alpha:

    print(
        "Decision: Reject H0 at the 5% significance level."
    )

    print(
        "There is statistically significant evidence "
        "of a difference in purchase rates."
    )

else:

    print(
        "Decision: Fail to reject H0 at the 5% significance level."
    )

    print(
        "There is insufficient evidence of a difference "
        "in purchase rates."
    )


# ============================================================
# PART C: PLACEBO TEST
# ============================================================
#
# We randomly shuffle the treatment assignment.
#
# This destroys the actual relationship between treatment
# and outcome.
#
# Therefore, the placebo treatment should produce an
# estimated effect close to zero.
# ============================================================

print("\n" + "=" * 65)
print("PART C: PLACEBO TEST")
print("=" * 65)


n_placebo = 500

placebo_ates = []


for i in range(n_placebo):

    # Randomly shuffle treatment
    placebo_T = rng.permutation(T)


    # Calculate placebo ATE
    placebo_treated = Y[
        placebo_T == 1
    ].mean()


    placebo_control = Y[
        placebo_T == 0
    ].mean()


    placebo_ate = (
        placebo_treated -
        placebo_control
    )


    placebo_ates.append(
        placebo_ate
    )


placebo_ates = np.array(
    placebo_ates
)


# ============================================================
# Placebo results
# ============================================================

placebo_mean = (
    placebo_ates.mean()
)

placebo_lower = np.percentile(
    placebo_ates,
    2.5
)

placebo_upper = np.percentile(
    placebo_ates,
    97.5
)


print(
    f"Mean placebo ATE: "
    f"{placebo_mean:.4f}"
)

print(
    f"95% placebo interval: "
    f"[{placebo_lower:.4f}, "
    f"{placebo_upper:.4f}]"
)


# ============================================================
# Placebo visualization
# ============================================================

plt.figure(
    figsize=(9, 6)
)

plt.hist(
    placebo_ates,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Zero effect"
)

plt.axvline(
    naive_ate,
    linestyle="-",
    linewidth=2,
    label=f"Observed ATE = {naive_ate:.4f}"
)

plt.xlabel(
    "Placebo ATE"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Placebo Treatment Effect Distribution"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "results/plots/08_placebo_test.png",
    dpi=300
)

plt.close()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("STEP 5 SUMMARY")
print("=" * 65)

print(
    f"True ATE:              {true_ate:.4f}"
)

print(
    f"Naive ATE:             {naive_ate:.4f}"
)

print(
    f"Bootstrap mean ATE:    {bootstrap_mean:.4f}"
)

print(
    f"95% Bootstrap CI:      "
    f"[{lower_ci:.4f}, {upper_ci:.4f}]"
)

print(
    f"Hypothesis test p-value: "
    f"{p_value:.6f}"
)

print(
    f"Mean placebo ATE:      "
    f"{placebo_mean:.4f}"
)

print(
    f"95% placebo interval:  "
    f"[{placebo_lower:.4f}, "
    f"{placebo_upper:.4f}]"
)

print("=" * 65)

print(
    "\nPlots saved to:"
)

print(
    "results/plots/07_bootstrap_ate.png"
)

print(
    "results/plots/08_placebo_test.png"
)

print("=" * 65)

STEP 5: STATISTICAL VALIDATION
Naive ATE: 0.0209
True ATE:  0.0150

PART A: BOOTSTRAP CONFIDENCE INTERVAL
Bootstrap replications: 500
Bootstrap mean ATE: 0.0205
95% Bootstrap CI: [0.0049, 0.0359]
The confidence interval does not contain zero.

PART B: HYPOTHESIS TEST
T-statistic: 2.5423
P-value: 0.011029
Decision: Reject H0 at the 5% significance level.
There is statistically significant evidence of a difference in purchase rates.

PART C: PLACEBO TEST
Mean placebo ATE: 0.0001
95% placebo interval: [-0.0161, 0.0172]

STEP 5 SUMMARY
True ATE:              0.0150
Naive ATE:             0.0209
Bootstrap mean ATE:    0.0205
95% Bootstrap CI:      [0.0049, 0.0359]
Hypothesis test p-value: 0.011029
Mean placebo ATE:      0.0001
95% placebo interval:  [-0.0161, 0.0172]

Plots saved to:
results/plots/07_bootstrap_ate.png
results/plots/08_placebo_test.png


In [ ]:
# ============================================================
# STEP 6: ROI & TARGETING STRATEGY
# ============================================================
#
# Objective:
# Translate causal treatment effects into business value.
#
# We compare:
# 1. Top 10% uplift targeting
# 2. Top 30% uplift targeting
# 3. Random 30% targeting
#
# BUSINESS ASSUMPTIONS
# --------------------
# Discount rate = 10%
#
# True individual treatment effect represents the incremental
# probability of purchase caused by the discount.
#
# Incremental purchases:
#       sum(true treatment effect)
#
# Incremental revenue:
#       sum(true treatment effect * average order value)
#
# Discount cost:
#       incremental purchases * average order value
#       * discount rate
#
# ROI:
#       (Incremental Revenue - Discount Cost)
#       / Discount Cost * 100
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. Create output directory
# ============================================================

os.makedirs(
    "results/plots",
    exist_ok=True
)


# ============================================================
# 2. Load Step 4 results
# ============================================================

data = pd.read_csv(
    "results/uplift_targeting_results.csv"
)


# ============================================================
# 3. Business assumptions
# ============================================================

DISCOUNT_RATE = 0.10

RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)


print("=" * 65)
print("STEP 6: ROI & TARGETING STRATEGY")
print("=" * 65)

print("\nBUSINESS ASSUMPTIONS")
print("-" * 65)

print(
    f"Discount rate: "
    f"{DISCOUNT_RATE:.0%}"
)

print(
    "Revenue per incremental purchase = "
    "Average Order Value"
)

print(
    "Discount cost is calculated only on "
    "incremental purchases."
)


# ============================================================
# 4. ROI calculation function
# ============================================================

def calculate_roi(customers):

    # --------------------------------------------------------
    # Incremental purchases
    # --------------------------------------------------------
    #
    # true_individual_effect is the increase in purchase
    # probability caused by the discount.
    #
    # Summing these probabilities gives expected incremental
    # purchases.
    # --------------------------------------------------------

    incremental_purchases = (
        customers[
            "true_individual_effect"
        ].sum()
    )


    # --------------------------------------------------------
    # Incremental revenue
    # --------------------------------------------------------
    #
    # For each customer:
    #
    # incremental probability
    # × average order value
    # --------------------------------------------------------

    incremental_revenue = (
        customers[
            "true_individual_effect"
        ]
        *
        customers[
            "avg_order_value"
        ]
    ).sum()


    # --------------------------------------------------------
    # Discount cost
    # --------------------------------------------------------
    #
    # Cost of the discount on incremental purchases only.
    # --------------------------------------------------------

    discount_cost = (
        customers[
            "true_individual_effect"
        ]
        *
        customers[
            "avg_order_value"
        ]
        *
        DISCOUNT_RATE
    ).sum()


    # --------------------------------------------------------
    # Net incremental value
    # --------------------------------------------------------

    net_incremental_value = (
        incremental_revenue
        -
        discount_cost
    )


    # --------------------------------------------------------
    # ROI
    # --------------------------------------------------------

    if discount_cost > 0:

        roi = (
            net_incremental_value
            /
            discount_cost
        ) * 100

    else:

        roi = np.nan


    return {
        "customers":
            len(customers),

        "incremental_purchases":
            incremental_purchases,

        "incremental_revenue":
            incremental_revenue,

        "discount_cost":
            discount_cost,

        "net_incremental_value":
            net_incremental_value,

        "ROI_percent":
            roi
    }


# ============================================================
# 5. Rank customers using predicted uplift
# ============================================================

data = data.sort_values(
    "predicted_uplift",
    ascending=False
).reset_index(
    drop=True
)


# ============================================================
# 6. TOP 10% TARGETING
# ============================================================

n_10 = int(
    0.10 * len(data)
)

top_10 = data.iloc[
    :n_10
]

result_10 = calculate_roi(
    top_10
)


# ============================================================
# 7. TOP 30% TARGETING
# ============================================================

n_30 = int(
    0.30 * len(data)
)

top_30 = data.iloc[
    :n_30
]

result_30 = calculate_roi(
    top_30
)


# ============================================================
# 8. RANDOM 30% TARGETING
# ============================================================

random_30 = data.sample(
    n=n_30,
    random_state=RANDOM_SEED
)

result_random = calculate_roi(
    random_30
)


# ============================================================
# 9. Create ROI comparison table
# ============================================================

roi_results = pd.DataFrame([
    {
        "strategy":
            "Top 10% Uplift Targeting",

        **result_10
    },

    {
        "strategy":
            "Top 30% Uplift Targeting",

        **result_30
    },

    {
        "strategy":
            "Random 30% Targeting",

        **result_random
    }
])


# ============================================================
# 10. Print ROI results
# ============================================================

print("\n" + "=" * 65)
print("ROI COMPARISON")
print("=" * 65)

print(
    roi_results.to_string(
        index=False,
        formatters={

            "incremental_purchases":
                "{:.2f}".format,

            "incremental_revenue":
                "{:.2f}".format,

            "discount_cost":
                "{:.2f}".format,

            "net_incremental_value":
                "{:.2f}".format,

            "ROI_percent":
                "{:.2f}%".format
        }
    )
)


# ============================================================
# 11. Compare Top 30% vs Random 30%
# ============================================================

top30_roi = (
    result_30[
        "ROI_percent"
    ]
)

random30_roi = (
    result_random[
        "ROI_percent"
    ]
)


roi_gain = (
    top30_roi -
    random30_roi
)


print("\n" + "=" * 65)
print("TARGETING STRATEGY COMPARISON")
print("=" * 65)

print(
    f"Top 30% uplift ROI: "
    f"{top30_roi:.2f}%"
)

print(
    f"Random 30% ROI: "
    f"{random30_roi:.2f}%"
)

print(
    f"ROI improvement: "
    f"{roi_gain:.2f} percentage points"
)


# ============================================================
# 12. Relative improvement
# ============================================================

if random30_roi != 0:

    relative_improvement = (
        (
            top30_roi -
            random30_roi
        )
        /
        abs(random30_roi)
    ) * 100

else:

    relative_improvement = np.nan


print(
    f"Relative ROI improvement: "
    f"{relative_improvement:.2f}%"
)


# ============================================================
# 13. Business interpretation
# ============================================================

print("\n" + "=" * 65)
print("BUSINESS INTERPRETATION")
print("=" * 65)


if top30_roi > random30_roi:

    print(
        "The uplift-based Top 30% strategy "
        "outperforms random targeting."
    )

    print(
        "This indicates that the uplift model "
        "provides some value for discount targeting."
    )

else:

    print(
        "The uplift-based Top 30% strategy "
        "does not outperform random targeting."
    )

    print(
        "The current model does not provide enough "
        "individual-level targeting value."
    )


# ============================================================
# 14. Additional business metrics
# ============================================================

top30_incremental_revenue = (
    result_30[
        "incremental_revenue"
    ]
)

random30_incremental_revenue = (
    result_random[
        "incremental_revenue"
    ]
)

revenue_gain = (
    top30_incremental_revenue
    -
    random30_incremental_revenue
)


print("\n" + "=" * 65)
print("INCREMENTAL REVENUE COMPARISON")
print("=" * 65)

print(
    f"Top 30% incremental revenue: "
    f"{top30_incremental_revenue:.2f}"
)

print(
    f"Random 30% incremental revenue: "
    f"{random30_incremental_revenue:.2f}"
)

print(
    f"Revenue gain from uplift targeting: "
    f"{revenue_gain:.2f}"
)


# ============================================================
# 15. ROI visualization
# ============================================================

plt.figure(
    figsize=(9, 6)
)

strategies = roi_results[
    "strategy"
]

roi_values = roi_results[
    "ROI_percent"
]

plt.bar(
    strategies,
    roi_values,
    edgecolor="black"
)

plt.ylabel(
    "ROI (%)"
)

plt.xlabel(
    "Targeting Strategy"
)

plt.title(
    "ROI Comparison of Discount Targeting Strategies"
)

plt.xticks(
    rotation=15
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.tight_layout()

plt.savefig(
    "results/plots/09_roi_comparison.png",
    dpi=300
)

plt.close()


# ============================================================
# 16. Incremental revenue visualization
# ============================================================

plt.figure(
    figsize=(9, 6)
)

revenue_values = [
    result_10[
        "incremental_revenue"
    ],

    result_30[
        "incremental_revenue"
    ],

    result_random[
        "incremental_revenue"
    ]
]

plt.bar(
    strategies,
    revenue_values,
    edgecolor="black"
)

plt.ylabel(
    "Expected Incremental Revenue"
)

plt.xlabel(
    "Targeting Strategy"
)

plt.title(
    "Expected Incremental Revenue by Targeting Strategy"
)

plt.xticks(
    rotation=15
)

plt.tight_layout()

plt.savefig(
    "results/plots/10_incremental_revenue.png",
    dpi=300
)

plt.close()


# ============================================================
# 17. Save ROI results
# ============================================================

roi_results.to_csv(
    "results/roi_targeting_results.csv",
    index=False
)


# ============================================================
# 18. Final output
# ============================================================

print("\n" + "=" * 65)
print("STEP 6 COMPLETED")
print("=" * 65)

print(
    "ROI results saved to:"
)

print(
    "results/roi_targeting_results.csv"
)

print("\nPlots saved to:")

print(
    "results/plots/09_roi_comparison.png"
)

print(
    "results/plots/10_incremental_revenue.png"
)

print("=" * 65)

STEP 6: ROI & TARGETING STRATEGY

BUSINESS ASSUMPTIONS
-----------------------------------------------------------------
Discount rate: 10%
Revenue per incremental purchase = Average Order Value
Discount cost is calculated only on incremental purchases.

ROI COMPARISON
                strategy  customers incremental_purchases incremental_revenue discount_cost net_incremental_value ROI_percent
Top 10% Uplift Targeting       1000                 15.46             5965.56        596.56               5369.00     900.00%
Top 30% Uplift Targeting       3000                 44.48            15368.59       1536.86              13831.73     900.00%
    Random 30% Targeting       3000                 45.30            12671.77       1267.18              11404.59     900.00%

TARGETING STRATEGY COMPARISON
Top 30% uplift ROI: 900.00%
Random 30% ROI: 900.00%
ROI improvement: -0.00 percentage points
Relative ROI improvement: -0.00%

BUSINESS INTERPRETATION
The uplift-based Top 30% strategy does not o

In [ ]:
import numpy as np, pandas as pd

def naive_ate(seed, n=10_000):
    np.random.seed(seed)
    pp  = np.random.randint(0, 50, n)
    aov = np.random.uniform(10, 500, n)
    dsl = np.random.randint(1, 365, n)   # drawn to keep the random stream identical
    em  = np.random.binomial(1, 0.35, n)
    pt  = np.clip(0.20 + 0.50 * aov / 500, 0.05, 0.90)
    T   = (np.random.random(n) < pt).astype(int)
    b   = np.clip(0.10 + 0.002*pp + 0.00015*aov + 0.05*em, 0.01, 0.80)
    raw = 0.005 + 0.00015*pp + 0.000015*aov + 0.002*em
    eff = raw * 0.015 / raw.mean()
    p1  = np.clip(b + eff, 0.01, 0.95)
    Y   = (np.random.random(n) < np.where(T == 1, p1, b)).astype(int)
    return Y[T == 1].mean() - Y[T == 0].mean(), eff.mean()

seeds = [42, 100, 200, 300, 400, 500, 600, 700, 800, 900]
res = pd.DataFrame([naive_ate(s) for s in seeds], columns=["naive", "true"], index=seeds)
res["bias"] = res.naive - res.true
print(res.round(4))
print("mean naive", res.naive.mean(), "SD", res.naive.std(),
      "mean bias", res.bias.mean(), "MAE", res.bias.abs().mean(),
      "max abs error", res.bias.abs().max())

      naive   true    bias
42   0.0209  0.015  0.0059
100  0.0151  0.015  0.0001
200  0.0323  0.015  0.0173
300  0.0301  0.015  0.0151
400  0.0361  0.015  0.0211
500  0.0233  0.015  0.0083
600  0.0304  0.015  0.0154
700  0.0117  0.015 -0.0033
800  0.0265  0.015  0.0115
900  0.0336  0.015  0.0186
mean naive 0.02600294792401091 SD 0.008127722604549201 mean bias 0.011002947924010908 MAE 0.011656792951878112 max abs error 0.021137941609799274
